In [1]:
!pip install -q rank_bm25

import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import re

print("⏳ Connecting to ChromaDB...")
chroma_client = chromadb.PersistentClient(path="./chroma_database")
collection = chroma_client.get_collection(name="healthcare_knowledge_base")

print("⏳ Loading BGE-M3 embedding model...")
embedding_model = SentenceTransformer("BAAI/bge-m3")

print("⏳ Building the BM25 Keyword Index...")
# 1. Fetch all documents currently stored in ChromaDB
all_data = collection.get(include=['documents', 'metadatas'])
all_ids = all_data['ids']
all_documents = all_data['documents']
all_metadatas = all_data['metadatas']

def tokenize(text):
    """A fast tokenizer that removes punctuation and lowercases the text for exact matching."""
    text = text.lower()
    return re.findall(r'\b\w+\b', text)

# 2. Tokenize every document in the database
tokenized_corpus = [tokenize(doc) for doc in all_documents]

# 3. Initialize the highly optimized BM25 engine
bm25_engine = BM25Okapi(tokenized_corpus)

print(f"✅ BM25 Engine successfully indexed {len(tokenized_corpus)} chunks!")

⏳ Connecting to ChromaDB...
⏳ Loading BGE-M3 embedding model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

⏳ Building the BM25 Keyword Index...
✅ BM25 Engine successfully indexed 8461 chunks!


In [11]:
# --- REVISED CELL 2: Core Retrieval, Auto-Routing & Fusion Engine ---

DISTANCE_THRESHOLD = 0.46  # Calibrated up to 0.46 to allow lexical variance

def dynamic_query_router(user_query):
    """
    Scans query text to enforce metadata constraints, preventing domain cross-bleed.
    """
    query_clean = user_query.lower()
    
    # Expanded Scheme triggers
    scheme_keywords = ["apply", "scheme", "yojana", "pmsbym", "jssk", "jsy", "pmmvy", "suman", "financial", "assistance", "benefit", "bpl", "sc/st", "eligible", "rupees", "cash"]
    if any(kw in query_clean for kw in scheme_keywords):
        return "Government Healthcare Scheme"
    
    # Expanded Disease triggers
    disease_keywords = ["symptom", "cause", "disease", "cancer", "anaemia", "anxiety", "bipolar", "virus", "infection", "bite", "infant", "spread", "tuberculosis", "doctor", "clinical", "health", "blood", "pain"]
    if any(kw in query_clean for kw in disease_keywords):
        return "Disease & Awareness"
        
    return None

def weighted_reciprocal_rank_fusion(dense_ranks, sparse_ranks, dense_weight=0.85, sparse_weight=0.15, k=60):
    """
    Combines ranks. Dense gets 85% authority to prevent BM25 from burying long documents.
    """
    rrf_scores = {}
    
    for rank, doc_id in enumerate(dense_ranks):
        if doc_id not in rrf_scores: rrf_scores[doc_id] = 0.0
        rrf_scores[doc_id] += dense_weight * (1.0 / (rank + k))
        
    for rank, doc_id in enumerate(sparse_ranks):
        if doc_id not in rrf_scores: rrf_scores[doc_id] = 0.0
        rrf_scores[doc_id] += sparse_weight * (1.0 / (rank + k))
        
    sorted_fused_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [item[0] for item in sorted_fused_results]

def evaluate_retrieve(user_query, n_results=5):
    category_filter = dynamic_query_router(user_query)
    where_clause = {"category": category_filter} if category_filter else None
    
    # Phase 1: Dense Retrieval
    query_vector = embedding_model.encode(user_query).tolist()
    chroma_results = collection.query(
        query_embeddings=[query_vector],
        n_results=20, 
        where=where_clause,
        include=['distances', 'documents', 'metadatas']
    )
    
    if not chroma_results['ids'] or not chroma_results['ids'][0]:
        return [], 1.0
        
    dense_ids = chroma_results['ids'][0]
    dense_distances = chroma_results['distances'][0]
    
    # 🚨 BUG FIX: Evaluate Guardrail BEFORE Fusion using the absolute best Semantic match
    best_semantic_distance = dense_distances[0]
    if best_semantic_distance > DISTANCE_THRESHOLD:
        return [], best_semantic_distance
    
    # Phase 2: Sparse Retrieval
    tokenized_query = tokenize(user_query)
    bm25_scores = bm25_engine.get_scores(tokenized_query)
    scored_docs = list(zip(bm25_scores, all_ids))
    scored_docs.sort(key=lambda x: x[0], reverse=True)
    
    if category_filter:
        valid_bm25_ids = [doc_id for score, doc_id in scored_docs if all_metadatas[all_ids.index(doc_id)].get('category') == category_filter][:20]
    else:
        valid_bm25_ids = [doc_id for score, doc_id in scored_docs][:20]
        
    # Phase 3: Weighted Fusion
    final_ranked_ids = weighted_reciprocal_rank_fusion(dense_ids, valid_bm25_ids)
    
    return final_ranked_ids[:n_results], best_semantic_distance

def hybrid_retrieve_and_verify(user_query, n_results=3):
    print(f"\n🔍 HYBRID SEARCH (85/15 routed): '{user_query}'")
    retrieved_ids, best_distance = evaluate_retrieve(user_query, n_results=n_results)
    
    if not retrieved_ids:
        print(f"🛑 VERIFICATION FAILED: Closest match failed safety check (Distance: {best_distance:.2f}).")
        print("🤖 Chatbot: 'I am sorry, but I do not have enough information in my database to answer your query.'")
        print("-" * 60)
        return

    print(f"✅ VERIFICATION PASSED! (Semantic Distance: {best_distance:.2f})")
    for i, doc_id in enumerate(retrieved_ids):
        idx = all_ids.index(doc_id)
        print(f"\n📌 Fused Rank #{i+1} [ID: {doc_id}]")
        print(f"📄 Source: {all_metadatas[idx]['source_name']} ({all_metadatas[idx]['title']})")
        print(f"📝 Text Snippet:\n{all_documents[idx]}")
    print("-" * 60)

In [12]:
test_queries = [
    "What are symptoms of diabetes?",
    "Who can apply for PM Kisan?", 
    "When should I see a doctor for dengue?",
    "What causes hypertension?",
    "How is tuberculosis spread?",
]

print("\n🚀 Running Hybrid Search Benchmark...")

for query in test_queries:
    hybrid_retrieve_and_verify(query, n_results=2)


🚀 Running Hybrid Search Benchmark...

🔍 HYBRID SEARCH (85/15 routed): 'What are symptoms of diabetes?'
✅ VERIFICATION PASSED! (Semantic Distance: 0.21)

📌 Fused Rank #1 [ID: who_diabetes_chunk_3]
📄 Source: WHO Fact Sheets (Diabetes)
📝 Text Snippet:
Section: Symptoms

Symptoms of diabetes may occur suddenly. In type 2 diabetes, the symptoms can be mild and may take many years to be noticed. Symptoms of diabetes include: - feeling very thirsty - needing to urinate more often than usual - blurred vision - feeling tired - losing weight unintentionally Over time, diabetes can damage blood vessels in the heart, eyes, kidneys and nerves. People with diabetes have a higher risk of health problems including heart attack, stroke and kidney failure. Diabetes can cause permanent vision loss by damaging blood vessels in the eyes. Many people with diabetes develop problems with their feet from nerve damage and poor blood flow. This can cause foot ulcers and may lead to amputation.

📌 Fused Rank #2 

In [13]:
# --- NEW CELL 5: Automated Phase 2 RAG Performance Evaluation Layer ---
import json

def run_pipeline_benchmark(test_set_path="golden_test_set.json"):
    print("🚀 Initiating Automated RAG Pipeline Benchmark Evaluation...")
    
    try:
        with open(test_set_path, 'r', encoding='utf-8') as f:
            evaluation_queries = json.load(f)
    except FileNotFoundError:
        print(f"❌ Error: Could not find '{test_set_path}' in your directory. Please save the JSON file first.")
        return
        
    total_queries = len(evaluation_queries)
    successful_recalls = 0
    strict_matches = 0
    false_positives = 0
    true_negatives = 0
    
    print(f"Loaded {total_queries} evaluation targets. Processing execution lines...\n" + "-"*70)
    
    for idx, item in enumerate(evaluation_queries):
        query = item['query']
        expected_ids = item['expected_chunk_ids']
        
        # Retrieve candidates
        retrieved_ids, winning_distance = evaluate_retrieve(query, n_results=5)
        
        # Scenario A: Out of Bounds query check
        if len(expected_ids) == 0:
            if len(retrieved_ids) == 0:
                true_negatives += 1
            else:
                false_positives += 1
                print(f"❌ Guardrail Leak on Test #{idx+1}: '{query}' brought up {retrieved_ids[0]}")
            continue
            
        # Scenario B: Valid Content retrieval check
        matched_elements = set(expected_ids).intersection(set(retrieved_ids))
        recall_score = len(matched_elements) / len(expected_ids)
        
        if recall_score > 0:
            successful_recalls += 1
        if recall_score == 1.0:
            strict_matches += 1
        else:
            print(f"⚠️ Partial/Zero Match on Test #{idx+1}: '{query}' \n    Expected: {expected_ids}\n    Got: {retrieved_ids}")

    # Calculate metrics
    content_queries_count = total_queries - (true_negatives + false_positives)
    guardrail_queries_count = true_negatives + false_positives
    
    recall_rate = (successful_recalls / content_queries_count) * 100 if content_queries_count > 0 else 0
    strict_accuracy = (strict_matches / content_queries_count) * 100 if content_queries_count > 0 else 0
    guardrail_accuracy = (true_negatives / guardrail_queries_count) * 100 if guardrail_queries_count > 0 else 0
    
    print("\n" + "="*70)
    print("📊 FINAL RETRIEVAL PIPELINE METRIC REPORT:")
    print("="*70)
    print(f"🔹 Chunk Recall@5 Rate      : {recall_rate:.2f}% (Found at least one target chunk)")
    print(f"🔹 Strict Multi-Chunk Accuracy: {strict_accuracy:.2f}% (Captured all needed context)")
    print(f"🔹 Guardrail Rejection Safety : {guardrail_accuracy:.2f}% (Blocked out-of-bounds queries cleanly)")
    print("="*70)

# Run the benchmark execution suite
run_pipeline_benchmark()

🚀 Initiating Automated RAG Pipeline Benchmark Evaluation...
Loaded 55 evaluation targets. Processing execution lines...
----------------------------------------------------------------------
⚠️ Partial/Zero Match on Test #1: 'What are the basic conditions for an applicant to be eligible for JSSK?' 
    Expected: ['gov_scheme_jssk_chunk_4']
    Got: ['gov_scheme_massy_chunk_4', 'gov_scheme_cpy_chunk_3', 'gov_scheme_mgbuy_chunk_4', 'gov_scheme_igndpsjak_chunk_4', 'gov_scheme_mpy_chunk_3']
⚠️ Partial/Zero Match on Test #2: 'How is an offline registration or referral handled for the Janani Shishu Suraksha Karyakram?' 
    Expected: ['gov_scheme_jssk_chunk_5']
    Got: ['gov_scheme_jsy1_chunk_16', 'gov_scheme_jsy1_chunk_11', 'gov_scheme_jsskmp_chunk_2', 'gov_scheme_jsy1_chunk_13', 'gov_scheme_jbsyb_chunk_12']
⚠️ Partial/Zero Match on Test #4: 'Do Above Poverty Line women in Low Performing States get any benefits under Janani Suraksha Yojana?' 
    Expected: ['gov_scheme_jsy1_chunk_9']
    G